# Exploration Notebook — The Fine Food Review Files

This notebook is the **interactive layer** of the project. The heavy lifting (cleaning, segmentation, sentiment, statistics) lives in the `python/` pipeline scripts — this notebook loads their exports and lets you poke at the results cell by cell, without re-running 40 minutes of work.

**Run the pipeline first** (see README §4), then: `uv run jupyter notebook notebooks/exploration.ipynb`

In [ ]:
import json
from pathlib import Path

import pandas as pd

EXPORTS = Path("../exports")

kpis = json.loads((EXPORTS / "kpis.json").read_text())
segments = pd.read_csv(EXPORTS / "segment_summary.csv")
monthly = pd.read_csv(EXPORTS / "monthly_activity.csv")
sentiment = pd.read_csv(EXPORTS / "sentiment_by_score.csv")
gaps = pd.read_csv(EXPORTS / "gap_summary.csv")
advocates = pd.read_csv(EXPORTS / "advocates.csv")
audit = json.loads((EXPORTS / "wrangling_audit.json").read_text())

print(f"Clean reviews: {kpis['clean_reviews']:,}  |  Customers: {kpis['customers']:,}")
print(f"Sentiment-rating agreement: {kpis['sentiment_rating_agreement_pct']}%")

## 1. The audit trail — what was removed and why

In [ ]:
pd.DataFrame(audit)[["step", "rule", "reason", "rows_removed", "rows_after"]]

## 2. Segment profiles — who are the six kinds of customer?

In [ ]:
segments.style.format({"pct_customers": "{:.1f}%", "avg_score": "{:.2f}",
                        "avg_recency_days": "{:.0f}"})

In [ ]:
segments.plot(kind="bar", x="segment", y="customers", legend=False, figsize=(8, 3), color="#2f4d6b");

## 3. Retention timeline — new vs. active reviewers

In [ ]:
ax = monthly.plot(x="month", y="new_users", figsize=(9, 3), color="#8f2b1e", label="new")
monthly.plot(x="month", y="active_users", ax=ax, color="#2f4d6b", label="active");

## 4. Sentiment validation — do the words match the stars?

In [ ]:
sentiment

In [ ]:
# The off-diagonal story: hidden detractors and hidden advocates
gaps

## 5. Advocates — the activation shortlist

In [ ]:
advocates.sort_values("advocacy_score", ascending=False).head(10)

## 6. Your sandbox

Everything above is read-only exploration. If you want to dig into the **review-level** data, load the big export (regenerate it with stage 2 if needed):

```python
reviews = pd.read_csv(EXPORTS / "reviews_scored.csv")
reviews.groupby("segment")["polarity"].describe()
```